In [1]:
from langchain_openrouter import ChatOpenRouter
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENROUTER_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("OPENROUTER_API_KEY not found")

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenRouter(model="openai/gpt-4o-mini", temperature=0)

Bro API KEY Variable exists


In [2]:

from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm.with_structured_output(llm_schema)

In [3]:

# TASK - 1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    ("human", "Please categorize the movie review as positive or negative : {input}")
])

In [4]:

# TASK - 2 [LLM with structured output]

llm_structured_output = llm.with_structured_output(llm_schema)

In [5]:
# TASK - 3 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def pydantic_json(input: llm_schema) -> str:
    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

In [7]:
# TASK - 1 [Prompt]
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")
])

# TASK - 2 [LLM]
llm_linkedin = ChatOpenRouter(model="openai/gpt-4o-mini", temperature=0)

# TASK - 3 [Str Parser]
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_linkedin | str_parser

In [8]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch

In [9]:
def insta_chain(text: dict):
    text = text["text"]

    # TASK - 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an Instagram post generator"),
        ("human", "Create a post for the following text for Instagram: {text}")
    ])

    # TASK - 2 [LLM]
    llm_insta = ChatOpenRouter(model="openai/gpt-4o-mini", temperature=0)

    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_insta | str_parser

    result = chain_insta.invoke({"text": text})

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

In [10]:
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
    insta_chain_runnable
)

final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

In [11]:

final_orchestrator.invoke({"input": "I loved this KGF movie"})

'🌟 Embracing Positivity in the Workplace! 🌟\n\nIn today\'s fast-paced world, it\'s easy to get caught up in challenges and setbacks. However, I\'ve found that maintaining a positive mindset can transform not only our own experiences but also the environment around us.\n\nHere are a few ways to cultivate positivity at work:\n\n1. **Celebrate Small Wins**: Every achievement, no matter how minor, deserves recognition. It boosts morale and encourages continued effort.\n\n2. **Practice Gratitude**: Take a moment each day to reflect on what you\'re thankful for. Acknowledging the good can shift your perspective and inspire those around you.\n\n3. **Support Each Other**: A simple "How can I help?" can go a long way. Let\'s lift each other up and create a culture of collaboration.\n\n4. **Stay Curious**: Embrace challenges as opportunities to learn. A positive attitude towards growth can lead to innovative solutions.\n\n5. **Lead by Example**: Positivity is contagious! By embodying a positive 